# Lab 4B: Closed-Loop PI Control

Run the cells in order. The MicroPython program remains active and waits for
commands after the serial connection is established.

In [ ]:
# PROVIDED CODE — DO NOT MODIFY THIS CELL.
from serial import Serial
from serial.tools import list_ports
import time
import numpy as np
from matplotlib import pyplot as plt

## 1. Open the Serial Connection

Keep the USB cable connected to the Shoe of Brian. In Thonny, record the serial
port shown for `MicroPython (generic)`, then switch the interpreter to
`Local Python 3` so that Jupyter can use the same port.

List the available ports and identify that Shoe USB port.

In [ ]:
# PROVIDED CODE — DO NOT MODIFY THIS CELL.
for port in list_ports.comports():
    print(port.device, port.description)

Set `SERIAL_PORT` to the Shoe USB port found above. This is the only value
to change in the serial-connection setup.

In [ ]:
# TODO: Enter the Shoe USB serial port.
SERIAL_PORT = "COM3"

# PROVIDED VALUE — DO NOT MODIFY.
BAUDRATE = 115200

Open the serial port, restart the MicroPython program, and wait until
`main.py` prints `READY LAB4B_SERIAL_V1`.

**Provided code — do not modify this cell.**

In [ ]:
# PROVIDED CODE — DO NOT MODIFY THIS CELL.
ser = Serial(SERIAL_PORT, baudrate=BAUDRATE, timeout=0.2)
time.sleep(0.3)
ser.reset_input_buffer()

# Ctrl-B returns to the normal REPL, Ctrl-C stops a running program,
# and Ctrl-D soft-resets MicroPython so main.py runs again.
ser.write(b"\x02")
time.sleep(0.1)
ser.write(b"\x03")
time.sleep(0.1)
ser.write(b"\x04")

deadline = time.time() + 5
ready = False

while time.time() < deadline and ready == False:
    line = ser.readline().decode().strip()

    if line == "READY LAB4B_SERIAL_V1":
        ready = True
        print(line)

if ready == False:
    print("ERR did not receive READY. Check that main.py is uploaded and has no errors.")
else:
    print("Serial port is open and main.py is running.")

## 2. Define `run_command()`

`run_command()` sends one complete command through the open serial connection
and reads the response through `END`. After responding, the MicroPython program
immediately waits for another command.

**Provided code — do not modify this cell.**

In [ ]:
# PROVIDED CODE — DO NOT MODIFY THIS CELL.
def run_command(command, timeout=15):
    lines = []

    ser.write((command + "\r\n").encode())

    deadline = time.time() + timeout
    finished = False

    while time.time() < deadline and finished == False:
        current_line = ser.readline().decode().strip()

        if current_line != "":
            lines.append(current_line)

            if current_line == "END":
                finished = True
            elif current_line.startswith("DATA,") == False:
                print(current_line)

    if finished == False:
        print("ERR no END before timeout")

    return lines

## 3. Define the Plotting Function

This provided function loads one saved data file and plots the measured velocity,
setpoint, total controller output, and the proportional and integral actions.

**Provided code — do not modify this cell.**

In [ ]:
# PROVIDED CODE — DO NOT MODIFY THIS CELL.
def plot_step(kp, ki, setpoint_rad_s, filename):
    data = np.genfromtxt(filename, delimiter=",")

    times = data[:, 0]
    velocities = data[:, 1]
    outputs = data[:, 2]
    p_actions = data[:, 3]
    i_actions = data[:, 4]

    fig, axes = plt.subplots(2, 1, figsize=(7, 7))

    axes[0].plot(times, velocities, label="Measured velocity")
    axes[0].axhline(
        setpoint_rad_s,
        color="black",
        linestyle="--",
        label="Setpoint",
    )
    axes[0].set_xlabel("Time [s]")
    axes[0].set_ylabel("Velocity [rad/s]")
    axes[0].set_title(f"Closed-Loop Response: Kp = {kp}, Ki = {ki}")
    axes[0].grid(True)
    axes[0].legend()

    axes[1].plot(times, p_actions, "--", label="P action")
    axes[1].plot(times, i_actions, "--", label="I action")
    axes[1].plot(times, outputs, label="Total output")
    axes[1].set_ylim(-110, 110)
    axes[1].set_xlabel("Time [s]")
    axes[1].set_ylabel("Motor voltage [%]")
    axes[1].set_title("Controller Output")
    axes[1].grid(True)
    axes[1].legend()

    plt.tight_layout()
    plt.show()

## 4. Set the Experiment Parameters

Enter the controller gains, velocity setpoint, output limit, test duration, and
a unique filename for this experiment. These are the values you will change
when comparing controllers.

In [ ]:
# TODO: Enter the experiment parameters specified in the lab manual.
KP = 0.1
KI = 0.0
SETPOINT_RAD_S = 500
OUTPUT_LIMIT_PERCENT = 100
TEST_TIME_MS = 1000
DATA_FILENAME = "data1.csv"

## 5. Run One Controller Experiment

The provided cell constructs a `RUN_CONTROLLER` command from the parameters
above. Do not edit the command phrase or communication code.

In [ ]:
# PROVIDED COMMAND CODE — DO NOT MODIFY THIS CELL.
response_lines = run_command(
    (
        f"RUN_CONTROLLER {KP} {KI} {SETPOINT_RAD_S} "
        f"{OUTPUT_LIMIT_PERCENT} {TEST_TIME_MS}"
    ),
    timeout=TEST_TIME_MS / 1000 + 10,
)

## 6. Save the Returned Data

Extract the `DATA` lines from the most recent response and save their five
numeric columns to the selected CSV file.

**Provided code — do not modify this cell.**

In [ ]:
# PROVIDED CODE — DO NOT MODIFY THIS CELL.
data_lines = [
    line[len("DATA,"):]
    for line in response_lines
    if line.startswith("DATA,")
]

if len(data_lines) == 0:
    print("ERR no data were returned by the microcontroller")
else:
    with open(DATA_FILENAME, "w") as data_file:
        for data_line in data_lines:
            data_file.write(data_line + "\n")

    print(f"Saved {len(data_lines)} samples to {DATA_FILENAME}.")

## 7. Plot the Results

Plot the most recently saved controller experiment.

**Provided code — do not modify this cell.**

In [ ]:
# PROVIDED CODE — DO NOT MODIFY THIS CELL.
plot_step(KP, KI, SETPOINT_RAD_S, DATA_FILENAME)

## 8. Repeat with Different Controller Gains

For each required test, change the parameter cell and use a new filename. Then
rerun the command, save-data, and plotting cells. The serial connection remains
open, so you do not need to rerun the connection setup.

## 9. Close the Serial Connection

After all tests are complete, close the serial port. You may then switch Thonny
back to `MicroPython (generic)`.

In [ ]:
# PROVIDED CODE — DO NOT MODIFY THIS CELL.
ser.close()